In [9]:
import pandas as pd
import random
import json

# --- CONFIGURATION ---
# Ensure this matches the exact name of the file you uploaded to Colab
INPUT_FILE = 'train.jsonl'
OUTPUT_FILE = 'final_real_world_dataset.csv'

# Define our volunteer skill categories (mapped to HumSet sectors)
skills_pool = {
    "Health": ["Emergency Nurse", "First Aid Specialist", "Medical Doctor", "Paramedic"],
    "WASH": ["Water Sanitation Engineer", "Plumbing Expert", "Well Driller"],
    "Education": ["Primary School Teacher", "Literacy Tutor", "ESL Instructor"],
    "Shelter": ["Construction Lead", "Carpenter", "Electrician"],
    "Logistics": ["Warehouse Manager", "Supply Chain Clerk", "Driver"]
}

# --- PROCESSING ---
print(f"📂 Reading {INPUT_FILE}...")

try:
    # Use lines=True because .jsonl is a 'JSON Lines' format
    df_raw = pd.read_json(INPUT_FILE, lines=True)
    print(f"✅ Successfully loaded {len(df_raw)} entries.")

    triplets = []
    # Shuffle the data to get a random mix
    df_shuffled = df_raw.sample(frac=1).reset_index(drop=True)

    TARGET_ROWS = 10000

    for _, row in df_shuffled.iterrows():
        if len(triplets) >= TARGET_ROWS:
            break

        # In HumSet 'train.jsonl', the text is in 'excerpt' and categories in 'sectors'
        anchor = str(row['excerpt']).replace('\n', ' ').strip()[:500]
        sectors = row['sectors']

        # Check if the sectors in this report match our skills_pool categories
        matched_cat = None
        for s in skills_pool.keys():
            # Check if our category name exists in the report's sector list
            if any(s.lower() in str(sec).lower() for sec in sectors):
                matched_cat = s
                break

        if matched_cat:
            # POSITIVE: A skill from the same category
            positive = random.choice(skills_pool[matched_cat])

            # NEGATIVE: A skill from a random DIFFERENT category
            other_cats = [c for c in skills_pool.keys() if c != matched_cat]
            negative = random.choice(skills_pool[random.choice(other_cats)])

            triplets.append({
                "anchor": anchor,
                "positive": positive,
                "negative": negative
            })

    # Save to CSV
    df_final = pd.DataFrame(triplets)
    df_final.to_csv(OUTPUT_FILE, index=False)

    print("-" * 30)
    print(f"🎉 DONE! Created {len(df_final)} rows in '{OUTPUT_FILE}'.")
    print("-" * 30)
    print(df_final.head(3))

except FileNotFoundError:
    print(f"❌ Error: File '{INPUT_FILE}' not found in Colab. Did you upload it?")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

📂 Reading train.jsonl...
✅ Successfully loaded 117435 entries.
------------------------------
🎉 DONE! Created 10000 rows in 'final_real_world_dataset.csv'.
------------------------------
                                              anchor         positive  \
0  [23 March 2021, Cox’s Bazar, Bangladesh] Situa...        Carpenter   
1  « Il y a toujours rupture des médicaments surt...  Emergency Nurse   
2  Remedial education for girls—and other childre...   Literacy Tutor   

          negative  
0  Plumbing Expert  
1           Driver  
2           Driver  
